# DuckDB v. Polars Benchmark

Polars and DuckDB are two newer programs providing more efficient processing. 

- polars:
  - https://pola.rs/
  - https://github.com/pola-rs/polars
  - polars is more of a data science replacement for the pandas library. It does really well with analytics and dataframe manipulation.

- duckdb:
  - https://duckdb.org/
  - https://github.com/duckdb/duckdb
  - duckdb is more of a sql technology, but it does have very fast aggregation and grouping of data

- benchmarks:
  - https://www.codecentric.de/en/knowledge-hub/blog/duckdb-vs-polars-performance-and-memory-with-massive-parquet-data
  - it was seen in the link above as well in the small sample analysis below that the query for a simple groupby were near identical for polars and duckdb. When doing a more complex aggregation, duckdb was much more efficient at 5-10x quicker.
  - the link also noted memory efficiencies which was not something confirmed in testing

- conclusion:
  - use duckdb for aggregating the data, but then use polars to analyze it

In [1]:
import os
import sys
import time

import duckdb
import polars as pl

os.chdir("../../")
sys.path.insert(0, os.getcwd())

In [2]:
from morai.experience import charters

In [3]:
# default is "plotly_mimetype+notebook", however that takes up space.
# "plotly_mimetype+notebook_connected" seems to save space
import plotly.io as pio

pio.renderers.default = "plotly_mimetype+notebook_connected"

## Data

In [4]:
pl_parquet_path = r"files/dataset/full_mortality_grouped_1223.parquet"

excl = [
    "amount_exposed",
    "policies_exposed",
    "death_claim_amount",
    "death_count",
    "cen2momp1wmi_byamt",
    "cen2momp2wmi_byamt",
    "qx_vbt15",
    "qx_raw",
    "qx_log_raw",
    "exp_amt_vbt15",
    "ae_vbt15",
]

## Polars

In [5]:
# Polars - simple groupby
t0 = time.time()
# reading in the dataset
lzdf = pl.scan_parquet(
    pl_parquet_path,
)
t1 = time.time()
fig = charters.chart(
    df=lzdf,
    x_axis="observation_year",
    y_axis="amount_exposed",
    type="line",
    display=False,
)
t2 = time.time()
print(f"Polars - Read: {t1-t0:.2f}s, Polars - Group: {t2-t1:.2f}s")

# polars - unique count
t0 = time.time()
schema = lzdf.collect_schema()
columns = list(schema.keys())
unique_counts = (
    lzdf.select(
        [pl.col(col).n_unique().alias(col) for col in columns if col not in excl]
    )
    .collect()
    .row(0, named=True)
)
t1 = time.time()
print(f"Polars - unique counts: {t1-t0:.2f}s")

Polars - Read: 0.00s, Polars - Group: 0.55s
Polars - unique counts: 10.62s


## DuckDB

In [8]:
t0 = time.time()
fig = duckdb.sql(
    f"""
    SELECT observation_year, SUM(amount_exposed) as amount_exposed
    FROM '{pl_parquet_path}'
    GROUP BY observation_year
    ORDER BY observation_year
"""
).pl()
t1 = time.time()
print(f"DuckDB - simple groupby: {t1-t0:.2f}s")

t0 = time.time()
columns = duckdb.sql(f"SELECT * FROM '{pl_parquet_path}' LIMIT 0").columns
columns = [col for col in columns if col not in excl]
counts_sql = ", ".join([f'COUNT(DISTINCT "{col}") as "{col}"' for col in columns])
unique_counts = (
    duckdb.sql(f"SELECT {counts_sql} FROM '{pl_parquet_path}'").pl().row(0, named=True)
)
t1 = time.time()
print(f"DuckDB - unique counts: {t1-t0:.2f}s")

DuckDB - simple groupby: 0.60s
DuckDB - unique counts: 1.19s
